# Tarea Hogar 04 — z496 corregido, en dos etapas (`z497`)

Archivo aparte de `z496` (`HT4960`). Experimento **`HT4970`**. La grilla es la misma de z496 (hojas chicas, `lr` bajo, `max_bin` alto, bagging bajo). Cambia cómo se mide, cómo se elige y cómo se entrena el final.

Lo que se arregla respecto de z496:

1. **Métrica.** Se deja de lado el umbral fijo `prob > 1/40`, que con undersampling < 1 queda descalibrado. Ahora se usa el **máximo de la curva de ganancia suavizada** en el holdout (media móvil de ±`ventana` clientes). Es la misma lógica de top-k que el submit por cupo. La ganancia con umbral queda como columna aparte para la planilla.
2. **El centro existe.** z496 buscaba `00_baseline`, que no existe; `delta_vs_centro` salía todo en NA.
3. **Dos etapas.** La media de 3 semillas sobre ~300 configs sigue eligiendo suerte, como en CazaTalentos. Las **top 10** se re-evalúan con **10 semillas nuevas**, distintas de las de la etapa 1. Se elige por esa media.
4. **El final usa todos los hiperparámetros del ganador.** Una sola función (`lgb_params_de`) arma los parámetros para el HPO y para el final. Antes `pos/neg_bagging`, `min_data_in_bin` y dart se perdían.
5. **`min_data_in_leaf` se reescala por `1 / (0.7 × undersampling)`.** El HPO entrena con el 70% del mes y el final con el 100%.
6. **Semillerío.** El modelo final promedia las probabilidades de `PARAM$semillerio` semillas.
7. **Resume por firma de config**, no por posición en la lista. Se puede editar la grilla sin que se salteen jobs equivocados.
8. **Cupo del submit.** Sale de la mediana del `k` óptimo suavizado de la etapa 2. No se elige mirando el Public.

**Paralelismo:** PSOCK con 1 thread por worker, igual que z495/z496.

## 0. Librerias

In [ ]:
# +++ lightgbm puede no estar en la imagen: se instala si falta
if (!require("data.table")) install.packages("data.table")
if (!require("lightgbm")) install.packages("lightgbm")
require("data.table")
require("lightgbm")
require("parallel")

setDTthreads(percent = 100)  # +++ el padre usa todos los cores para fread; los workers van a 1
options(scipen = 999)

## 1. PARAM

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 271211L   # +++ TU semilla
PARAM$estudiante <- "Maceo, Marcos"
PARAM$experimento <- "HT4970"
PARAM$mc_cores <- max(1L, detectCores() - 1L)  # +++ 7 workers en e2-highmem-8

PARAM$test_frac <- 0.30
PARAM$ventana <- 200L          # +++ suavizado de la curva de ganancia: media movil de +-200 clientes del holdout
PARAM$semillas_etapa1 <- c(271211L, 200177L, 410551L)
PARAM$top_etapa2 <- 10L
# +++ semillas NUEVAS para la etapa 2: no se reusan las que eligieron el top
PARAM$semillas_etapa2 <- c(552581L, 892237L, 123457L, 654323L, 777011L,
                           100703L, 300007L, 500009L, 700001L, 900007L)
PARAM$semillerio <- 10L        # +++ modelos promediados en el final
PARAM$mc_cores

## 2. Dataset (solo 202107: ahi hay clase)

In [ ]:
# +++ deteccion de entorno: Colab / VM GCP / local
candidatos_exp <- c("/content/buckets/b1/exp", path.expand("~/buckets/b1/exp"), file.path(getwd(), "exp"))
base_exp <- candidatos_exp[dir.exists(candidatos_exp)][1]
if (is.na(base_exp)) {
  base_exp <- candidatos_exp[3]
  dir.create(base_exp, recursive = TRUE, showWarnings = FALSE)
}
dir.create(file.path(base_exp, PARAM$experimento), showWarnings = FALSE)
setwd(file.path(base_exp, PARAM$experimento))
getwd()

candidatos_ds <- c(
  "/content/datasets/dataset_pequeno.csv",
  path.expand("~/datasets/dataset_pequeno.csv"),
  path.expand("~/buckets/b1/datasets/dataset_pequeno.csv")
)
archivo_dataset <- candidatos_ds[file.exists(candidatos_ds)][1]
stopifnot(!is.na(archivo_dataset))

dataset <- fread(archivo_dataset)
# +++ solo el mes con clase. 202109 queda para el submit final, no para elegir hiperparametros
dataset_mes <- dataset[foto_mes == 202107]
dataset_mes[, c("clase01", "azar", "training") := NULL]
nrow(dataset_mes)
dataset_mes[, .N, clase_ternaria]

## 3. Un job

Entrena en el 70% (con undersampling de CONTINUA) y predice el 30%. La **ganancia** es el máximo de la curva de ganancia acumulada suavizada, ordenando por probabilidad. Se escala al mes completo (`/0.30`). `mejores_envios` es el `k` de ese máximo, también escalado al mes.

`lgb_params_de()` arma los parámetros de LightGBM desde un job. La usan el HPO y el final, así el modelo que se sube es el que se evaluó.

In [ ]:
# +++ UNA sola funcion arma los params: la usan eval_job (HPO) y el final. Nada se pierde en el camino.
lgb_params_de <- function(job, num_threads = 1L, min_data_in_leaf = job$min_data_in_leaf, seed = job$seed) {
  params <- list(
    objective = job$objective,
    metric = "auc",
    boosting = job$boosting,
    num_threads = as.integer(num_threads),
    seed = as.integer(seed),
    verbosity = -1,
    learning_rate = job$learning_rate,
    num_leaves = as.integer(job$num_leaves),
    max_depth = as.integer(job$max_depth),
    min_data_in_leaf = as.integer(min_data_in_leaf),
    feature_fraction = job$feature_fraction,
    feature_fraction_bynode = job$feature_fraction_bynode,
    bagging_fraction = job$bagging_fraction,
    bagging_freq = as.integer(job$bagging_freq),
    lambda_l1 = job$lambda_l1,
    lambda_l2 = job$lambda_l2,
    min_gain_to_split = job$min_gain_to_split,
    min_sum_hessian_in_leaf = job$min_sum_hessian_in_leaf,
    scale_pos_weight = job$scale_pos_weight,
    is_unbalance = isTRUE(as.logical(job$is_unbalance)),
    boost_from_average = isTRUE(as.logical(job$boost_from_average)),
    extra_trees = isTRUE(as.logical(job$extra_trees)),
    path_smooth = job$path_smooth,
    first_metric_only = TRUE,
    feature_pre_filter = FALSE
  )
  if (isTRUE(as.logical(job$force_col_wise))) {
    params$force_col_wise <- TRUE
  } else {
    params$force_row_wise <- TRUE
  }
  if (identical(job$boosting, "dart")) {
    params$drop_rate <- job$drop_rate
    params$skip_drop <- job$skip_drop
    params$max_drop <- as.integer(job$max_drop)
  }
  if (job$pos_bagging_fraction < 1 || job$neg_bagging_fraction < 1) {
    params$pos_bagging_fraction <- job$pos_bagging_fraction
    params$neg_bagging_fraction <- job$neg_bagging_fraction
    if (params$bagging_freq == 0L) params$bagging_freq <- 1L
  }
  params
}

# +++ params del Dataset: max_bin y min_data_in_bin se fijan al construirlo
lgb_dataset_params_de <- function(job) {
  list(
    max_bin = as.integer(job$max_bin),
    min_data_in_bin = as.integer(job$min_data_in_bin),
    feature_pre_filter = FALSE
  )
}

# +++ maximo de la curva de ganancia suavizada (top-k, igual que el submit por cupo)
ganancia_suavizada <- function(prob, clase, ventana, test_frac) {
  ord <- order(prob, decreasing = TRUE)
  cs <- cumsum(ifelse(clase[ord] == "BAJA+2", 975000, -25000))
  suave <- data.table::frollmean(cs, 2L * ventana + 1L, align = "center")
  i <- which.max(suave)
  if (length(i) == 0L) i <- which.max(cs)       # holdout mas chico que la ventana
  list(ganancia = suave[i] / test_frac, k = i, k_full = as.integer(round(i / test_frac)))
}

# +++ un job = un modelo LightGBM. PSOCK: usa solo lo exportado (dataset_mes, PARAM y las 3 funciones de arriba).
eval_job <- function(job) {
  tryCatch({
    data.table::setDTthreads(1)
    t0 <- Sys.time()

    split_strat <- function(dt, p, seed) {
      set.seed(as.integer(seed))
      dwork <- data.table::copy(dt)
      dwork[, `:=`(azar = runif(.N), .rowid = seq_len(.N))]
      data.table::setorderv(dwork, c("clase_ternaria", "azar"))
      dwork[, fold := seq_len(.N) / .N, by = clase_ternaria]
      ids <- dwork[fold <= p, .rowid]
      list(
        train = dt[ids],
        test = dt[setdiff(seq_len(nrow(dt)), ids)]
      )
    }

    sp <- split_strat(dataset_mes, 1 - PARAM$test_frac, job$semilla_split)
    dtrain <- sp$train
    dtest <- sp$test

    set.seed(as.integer(job$semilla2))
    dtrain[, azar_us := runif(.N)]
    dfit <- dtrain[clase_ternaria %in% c("BAJA+1", "BAJA+2") | azar_us <= job$undersampling]

    lab <- function(cl) {
      if (identical(job$label_mode, "baja1y2")) {
        as.integer(cl %in% c("BAJA+1", "BAJA+2"))
      } else {
        as.integer(cl == "BAJA+2")
      }
    }

    drop_cols <- c(
      "clase_ternaria", "numero_de_cliente", "foto_mes",
      "azar", "azar_us", "fold", ".rowid"
    )
    campos <- setdiff(colnames(dfit), drop_cols)
    num_ok <- vapply(dfit[, campos, with = FALSE], is.numeric, logical(1))
    campos <- campos[num_ok]

    ds <- lightgbm::lgb.Dataset(
      data = data.matrix(dfit[, campos, with = FALSE]),
      label = lab(dfit$clase_ternaria),
      params = lgb_dataset_params_de(job),
      free_raw_data = TRUE
    )
    modelo <- lightgbm::lgb.train(
      params = lgb_params_de(job),
      data = ds,
      nrounds = as.integer(job$num_iterations),
      verbose = -1
    )

    prob <- predict(modelo, data.matrix(dtest[, campos, with = FALSE]))
    clase <- dtest$clase_ternaria
    gs <- ganancia_suavizada(prob, clase, PARAM$ventana, PARAM$test_frac)
    gan_umbral <- sum(ifelse(prob > (1 / 40), ifelse(clase == "BAJA+2", 975000, -25000), 0)) / PARAM$test_frac

    data.table::data.table(
      sig = job$sig,
      cfg_sig = job$cfg_sig,
      etapa = job$etapa,
      familia = job$familia,
      param_optim = job$param_optim,
      semilla_split = job$semilla_split,
      semilla2 = job$semilla2,
      seed = job$seed,
      ganancia = gs$ganancia,
      ganancia_umbral = gan_umbral,
      k_star = gs$k,
      mejores_envios = gs$k_full,
      tiempo_seg = round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1),
      error = ""
    )
  }, error = function(e) {
    data.table::data.table(
      sig = job$sig,
      cfg_sig = job$cfg_sig,
      etapa = job$etapa,
      familia = job$familia,
      param_optim = job$param_optim,
      ganancia = NA_real_,
      error = conditionMessage(e)
    )
  })
}

## 4. Los experimentos

Centro nuevo: hojas 8, `lr=0.027`, 1000 rondas, `min_data=76`, `max_bin=127`, undersampling 1. Cada config x 3 semillas fijas. El rankeo (seccion 6) usa la media.

In [ ]:
# +++ centro del diseno. Cada familia pisa UN eje (o un par acoplado) sobre este base.
# +++ No es el grid de z494 (AUC, 3 params). La metrica la calcula eval_job.
base <- list(
  boosting = "gbdt",
  objective = "binary",
  boost_from_average = TRUE,
  force_col_wise = FALSE,
  is_unbalance = FALSE,
  extra_trees = FALSE,
  num_iterations = 1000L,
  learning_rate = 0.027,
  feature_fraction = 0.8,
  feature_fraction_bynode = 1.0,
  min_data_in_leaf = 76L,
  num_leaves = 8L,
  max_depth = -1L,
  lambda_l1 = 0,
  lambda_l2 = 0,
  min_gain_to_split = 0,
  bagging_fraction = 1.0,
  bagging_freq = 0L,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  min_sum_hessian_in_leaf = 0.001,
  scale_pos_weight = 1,
  drop_rate = 0.1,
  skip_drop = 0.5,
  max_drop = 50L,
  max_bin = 127L,
  min_data_in_bin = 3L,
  path_smooth = 0,
  early_stopping_rounds = 0L,
  undersampling = 1.0,
  label_mode = "baja2",
  semilla_split = PARAM$semilla_primigenia,
  semilla2 = PARAM$semilla_primigenia + 17L,
  seed = PARAM$semilla_primigenia
)

fix_job <- function(job) {
  if (isTRUE(job$bagging_fraction < 1) && job$bagging_freq == 0L) job$bagging_freq <- 1L
  if (isTRUE(job$pos_bagging_fraction < 1 || job$neg_bagging_fraction < 1) && job$bagging_freq == 0L) {
    job$bagging_freq <- 1L
  }
  if (isTRUE(job$is_unbalance)) job$scale_pos_weight <- 1
  if (job$max_depth > 0 && job$num_leaves > (2^job$max_depth - 1)) {
    job$num_leaves <- as.integer(2^job$max_depth - 1)
  }
  if (identical(job$boosting, "dart") && job$num_iterations > 200L) {
    job$num_iterations <- 200L  # +++ dart es mucho mas lento; tope para que el barrido termine
  }
  job
}

one <- function(familia, param_optim, overrides) {
  job <- modifyList(base, overrides)
  job$familia <- familia
  job$param_optim <- param_optim
  fix_job(job)
}

expand_jobs <- function(familia, param_optim, grid) {
  tb <- do.call(CJ, grid)
  lapply(seq_len(nrow(tb)), function(i) {
    ov <- as.list(tb[i])
    one(familia, param_optim, ov)
  })
}

diagonal <- function(familia, param_optim, cols) {
  n <- length(cols[[1]])
  lapply(seq_len(n), function(i) {
    ov <- lapply(cols, function(v) v[[i]])
    one(familia, param_optim, ov)
  })
}

jobs <- list()
push <- function(xs) jobs <<- c(jobs, xs)

# --- referencias (puntos nombrados, no un barrido) ---
push(list(one(
  "00_baseline", "baseline del diseno",
  list()
)))
push(list(one(
  "00_ref_z102", "punto z102 (curso)",
  list(learning_rate = 0.05, num_iterations = 100L, num_leaves = 31L,
       min_data_in_leaf = 100L, feature_fraction = 0.5, max_bin = 31L, undersampling = 1)
)))
push(list(one(
  "00_ref_denicolay", "punto Denicolay planilla",
  list(num_iterations = 1000L, learning_rate = 0.027, feature_fraction = 0.8,
       min_data_in_leaf = 76L, num_leaves = 8L, max_depth = -1L, undersampling = 1)
)))

# --- centro = zona que en z495 no era ruido: pocas hojas, lr chico, datos completos ---
# +++ el 618M de "semilla" era el mismo modelo con otro split. Aca cada config se corre
# +++ en las MISMAS 3 semillas y se rankea por el promedio.

push(list(one("00_centro", "centro Denicolay empujado (us=1, leaves=8, lr=0.027, bin=127)", list())))
push(list(one(
  "00_ref_kaggle5940_style", "referencia: hojas medias, us=1, bin=31 (estilo del 375.9)",
  list(num_leaves = 31L, learning_rate = 0.05, num_iterations = 500L,
       min_data_in_leaf = 100L, feature_fraction = 0.8, max_bin = 31L,
       undersampling = 1, max_depth = -1L)
)))

# extremos de complejidad, alrededor de hojas chicas
push(expand_jobs("num_leaves", "num_leaves", list(
  num_leaves = c(2L, 3L, 4L, 6L, 8L, 12L, 16L, 24L, 32L, 48L, 64L)
)))
push(expand_jobs("min_data_in_leaf", "min_data_in_leaf", list(
  min_data_in_leaf = c(5L, 15L, 30L, 50L, 60L, 76L, 90L, 120L, 180L, 300L, 600L, 1200L)
)))
push(expand_jobs("max_depth", "max_depth", list(
  max_depth = c(-1L, 2L, 3L, 4L, 5L, 6L, 8L, 12L)
)))
push(expand_jobs("min_gain_to_split", "min_gain_to_split", list(
  min_gain_to_split = c(0, 0.01, 0.1, 1, 5, 20, 50)
)))
push(expand_jobs("min_sum_hessian_in_leaf", "min_sum_hessian_in_leaf", list(
  min_sum_hessian_in_leaf = c(1e-3, 0.1, 1, 10, 50, 200)
)))

# lr mas chico y muchas mas rondas (el extremo que Denicolay ya insinuaba)
push(diagonal(
  "lr_x_nrounds",
  "learning_rate + num_iterations extremos",
  list(
    learning_rate = c(0.003, 0.005, 0.01, 0.015, 0.02, 0.027, 0.04, 0.06, 0.1),
    num_iterations = c(5000L, 3000L, 2000L, 1500L, 1200L, 1000L, 600L, 350L, 200L)
  )
))
push(expand_jobs("num_iterations", "num_iterations (lr fijo 0.02)", list(
  learning_rate = c(0.02),
  num_iterations = c(400L, 800L, 1500L, 2500L, 4000L)
)))

# feature_fraction: el joint bueno uso 0.3 y 0.9; empujar ambos lados
push(expand_jobs("feature_fraction", "feature_fraction", list(
  feature_fraction = c(0.08, 0.15, 0.25, 0.4, 0.6, 0.8, 0.9, 0.97, 1.0)
)))
push(expand_jobs("feature_fraction_bynode", "feature_fraction_bynode", list(
  feature_fraction_bynode = c(0.2, 0.4, 0.6, 0.8, 1.0)
)))

# regularizacion mas alta
push(expand_jobs("lambda_l1", "lambda_l1", list(
  lambda_l1 = c(0, 0.1, 1, 3, 10, 30, 100)
)))
push(expand_jobs("lambda_l2", "lambda_l2", list(
  lambda_l2 = c(0, 0.1, 1, 10, 50, 200)
)))

# bagging fue la familia ESTABLE en z495 (media alta, rango chico). Empujar fraction chica.
push(expand_jobs("bagging", "bagging_fraction + bagging_freq", list(
  bagging_fraction = c(0.1, 0.2, 0.3, 0.45, 0.6, 0.8),
  bagging_freq = c(1L, 5L)
)))
push(expand_jobs("pos_neg_bagging", "pos/neg bagging", list(
  pos_bagging_fraction = c(1.0, 0.6),
  neg_bagging_fraction = c(0.15, 0.3, 0.5),
  bagging_freq = c(1L)
)))

# desbalance: is_unbalance en z495 destruyo (122M). No se repite.
# undersampling: los trials buenos tenian 1.0, no 0.02
push(expand_jobs("undersampling", "undersampling (lado alto)", list(
  undersampling = c(0.4, 0.6, 0.8, 0.9, 1.0)
)))
push(expand_jobs("scale_pos_weight", "scale_pos_weight (rango corto)", list(
  scale_pos_weight = c(1, 2, 4, 8)
)))

# max_bin 255 aparecio varias veces en el top. Subir hasta 1023.
push(expand_jobs("max_bin", "max_bin", list(
  max_bin = c(31L, 63L, 127L, 255L, 511L, 1023L)
)))
push(expand_jobs("min_data_in_bin", "min_data_in_bin", list(
  min_data_in_bin = c(1L, 3L, 10L, 30L, 80L)
)))

push(expand_jobs("extra_trees", "extra_trees", list(extra_trees = c(TRUE))))
push(expand_jobs("max_depth_con_hojas_chicas", "max_depth con num_leaves=8", list(
  max_depth = c(3L, 4L, 6L, 10L, -1L),
  num_leaves = c(8L)
)))

# interacciones en la zona que pago
push(expand_jobs("ix_leaves_x_mindata", "num_leaves x min_data", list(
  num_leaves = c(4L, 8L, 16L, 32L),
  min_data_in_leaf = c(30L, 76L, 150L, 400L)
)))
push(expand_jobs("ix_leaves_x_bin", "num_leaves x max_bin", list(
  num_leaves = c(4L, 8L, 16L),
  max_bin = c(63L, 255L, 511L)
)))
push(expand_jobs("ix_ff_x_leaves", "feature_fraction x num_leaves", list(
  feature_fraction = c(0.2, 0.4, 0.8, 1.0),
  num_leaves = c(4L, 8L, 16L, 32L)
)))
push(expand_jobs("ix_bag_x_leaves", "bagging x num_leaves", list(
  bagging_fraction = c(0.2, 0.5, 0.8),
  bagging_freq = c(1L),
  num_leaves = c(8L, 16L, 32L)
)))
push(expand_jobs("ix_l1_x_leaves", "lambda_l1 x num_leaves", list(
  lambda_l1 = c(0, 1, 10),
  num_leaves = c(4L, 8L, 16L)
)))
push(diagonal(
  "ix_lr_largo",
  "lr muy chico x muchas rondas x hojas chicas",
  list(
    learning_rate = c(0.005, 0.01, 0.015, 0.02),
    num_iterations = c(4000L, 2500L, 2000L, 1500L),
    num_leaves = c(4L, 8L, 8L, 12L),
    max_bin = c(255L, 255L, 127L, 255L)
  )
))

# conjunta SOLO en la caja que no exploto en z495
sample_joint <- function(n, familia, param_optim, seed) {
  set.seed(seed)
  out <- vector("list", n)
  for (i in seq_len(n)) {
    md <- sample(c(-1L, 3L, 4L, 6L, 8L), 1)
    nl <- sample(c(3L, 4L, 6L, 8L, 12L, 16L, 24L), 1)
    if (md > 0) nl <- min(nl, as.integer(2^md - 1))
    bf <- sample(c(1, 0.8, 0.5, 0.3, 0.15), 1)
    out[[i]] <- one(familia, param_optim, list(
      learning_rate = round(10^runif(1, log10(0.005), log10(0.05)), 4),
      num_iterations = sample(c(800L, 1200L, 1800L, 2500L, 4000L), 1),
      num_leaves = nl,
      max_depth = md,
      min_data_in_leaf = sample(c(20L, 40L, 60L, 76L, 100L, 150L, 250L), 1),
      feature_fraction = sample(c(0.15, 0.3, 0.5, 0.7, 0.85, 1), 1),
      lambda_l1 = sample(c(0, 0, 0.1, 1, 5), 1),
      lambda_l2 = sample(c(0, 0, 1, 5), 1),
      bagging_fraction = bf,
      bagging_freq = if (bf < 1) 1L else 0L,
      undersampling = sample(c(0.8, 1, 1, 1), 1),
      max_bin = sample(c(63L, 127L, 255L, 511L), 1),
      scale_pos_weight = 1
    ))
  }
  out
}
push(sample_joint(100L, "joint_extremo", "joint en la caja ganadora, empujada", PARAM$semilla_primigenia + 3L))

# +++ duplicados fuera. cfg_sig = firma de la config SIN semillas; sig = cfg_sig + semillas (clave del resume)
sig_of <- function(job) {
  nms <- sort(setdiff(names(job), c("trial_id", "familia", "param_optim", "semilla_split", "semilla2", "seed",
                                    "sig", "cfg_sig", "etapa")))
  paste(vapply(nms, function(nm) paste0(nm, "=", paste(job[[nm]], collapse = ",")), character(1)), collapse = "|")
}
configs <- list()
for (job in jobs) {
  s <- sig_of(job)
  if (!is.null(configs[[s]])) next
  job$cfg_sig <- s
  configs[[s]] <- job
}

con_semillas <- function(cfgs, semillas, etapa) {
  out <- list()
  for (job in cfgs) {
    for (sv in semillas) {
      j <- job
      j$semilla_split <- sv
      j$seed <- sv
      j$semilla2 <- sv + 17L
      j$etapa <- etapa
      j$sig <- paste0(j$cfg_sig, "#", sv)
      out[[length(out) + 1L]] <- j
    }
  }
  out
}

# +++ el centro se identifica por firma (el nombre de familia depende de quien aparecio primero)
centro_sig <- sig_of(fix_job(base))
stopifnot(centro_sig %in% names(configs))
configs[[centro_sig]]$familia <- "00_centro"

jobs <- con_semillas(configs, PARAM$semillas_etapa1, 1L)
cat("configs:", length(configs), "  jobs etapa 1 (x", length(PARAM$semillas_etapa1), " semillas): ", length(jobs), "\n", sep = "")
print(rbindlist(lapply(configs, function(j) data.table(familia = j$familia)))[, .N, by = familia][order(-N)])

## 5. Corrida (etapa 1)

PSOCK + 1 thread por worker. Progreso por tanda. Si un modelo se cae queda en `error` y el resto sigue.

El resume usa `sig` (config + semilla), no la posicion del job en la lista: si se agrega o saca una familia, lo ya corrido se reusa igual.

In [ ]:
correr <- function(jobs, archivo) {
  tb <- data.table()
  hechos <- character()
  if (file.exists(archivo)) {
    tb <- fread(archivo, sep = "\t")
    if ("ganancia" %in% names(tb)) hechos <- tb[is.finite(ganancia), unique(sig)]
    tb <- tb[sig %in% hechos]              # +++ los que fallaron se reintentan
  }
  pending <- Filter(function(j) !(j$sig %in% hechos), jobs)
  cat(archivo, "| pendientes:", length(pending), "de", length(jobs), "\n")
  if (length(pending) == 0L) return(tb)

  cl <- makeCluster(PARAM$mc_cores, type = "PSOCK")
  on.exit(stopCluster(cl), add = TRUE)
  clusterEvalQ(cl, {
    Sys.setenv(OMP_NUM_THREADS = "1", MKL_NUM_THREADS = "1")
    suppressPackageStartupMessages({
      library(data.table)
      library(lightgbm)
    })
    data.table::setDTthreads(1)
    NULL
  })
  clusterExport(cl, c("eval_job", "lgb_params_de", "lgb_dataset_params_de", "ganancia_suavizada",
                      "PARAM", "dataset_mes"), envir = .GlobalEnv)

  bs <- PARAM$mc_cores
  n_b <- ceiling(length(pending) / bs)
  t0 <- Sys.time()
  for (b in seq_len(n_b)) {
    idx <- ((b - 1L) * bs + 1L):min(b * bs, length(pending))
    tb_new <- rbindlist(parLapply(cl, pending[idx], eval_job), fill = TRUE)
    tb <- rbindlist(list(tb, tb_new), fill = TRUE)
    fwrite(tb, archivo, sep = "\t")

    n_ok <- tb_new[is.finite(ganancia), .N]
    mins <- as.numeric(difftime(Sys.time(), t0, units = "mins"))
    cat(sprintf(
      "tanda %d/%d | ok %d | fallos %d | mejor tanda %s | %.1f min | ETA %.1f min\n",
      b, n_b, n_ok, nrow(tb_new) - n_ok,
      if (n_ok) format(max(tb_new$ganancia, na.rm = TRUE), big.mark = ",", scientific = FALSE) else "NA",
      mins, if (b < n_b) mins / b * (n_b - b) else 0
    ))
    malos <- tb_new[!is.finite(ganancia)]
    if (nrow(malos)) print(malos[, .(familia, error)])
    flush.console()
  }
  tb
}

tb_trials <- correr(jobs, "trials_th04_etapa1.tsv")
cat("trials con ganancia:", tb_trials[is.finite(ganancia), .N], "\n")

## 6. Etapa 1: que parametro movio la ganancia

Se agrupa por `cfg_sig` (la config sin semillas) y se promedian las 3 semillas. `delta_vs_centro` > 0: esa familia encontro algo mejor que el centro. Si el rango de una familia es chico, ese parametro no vale la pena tunearlo.

In [ ]:
tb <- tb_trials[is.finite(ganancia)]
agg1 <- tb[, .(
  ganancia = mean(ganancia),
  ganancia_sd = sd(ganancia),
  n_seeds = .N,
  mejores_envios = as.integer(round(median(mejores_envios)))
), by = .(cfg_sig, familia, param_optim)]
agg1 <- agg1[n_seeds == length(PARAM$semillas_etapa1)]   # +++ solo configs completas

centro <- agg1[cfg_sig == centro_sig, ganancia][1]
agg1[, delta_vs_centro := ganancia - centro]
setorder(agg1, -ganancia)
fwrite(agg1, "configs_th04_etapa1.tsv", sep = "\t")

cat("centro (media", length(PARAM$semillas_etapa1), "semillas):", format(centro, big.mark = ",", scientific = FALSE), "\n\n")
cat("=== top 20 configs, etapa 1 ===\n")
print(agg1[1:min(20, .N), .(familia, ganancia, ganancia_sd, mejores_envios, delta_vs_centro)])

por_familia <- agg1[, .(
  n = .N,
  ganancia_max = max(ganancia),
  ganancia_mean = mean(ganancia)
), by = .(familia, param_optim)]
por_familia[, delta_vs_centro := ganancia_max - centro]
setorder(por_familia, -ganancia_max)
fwrite(por_familia, "sensibilidad_th04.tsv", sep = "\t")
cat("\n=== mejor config de cada familia ===\n")
print(por_familia)

## 7. Etapa 2: re-evaluar el top con semillas nuevas

El maximo de la etapa 1 esta sesgado para arriba (se eligio entre cientos de configs con las mismas 3 semillas). Las `top_etapa2` mejores, mas el centro, se corren de nuevo con 10 semillas que no se usaron para elegir. Se elige por esa media.

In [ ]:
top_sigs <- unique(c(agg1[1:min(PARAM$top_etapa2, .N), cfg_sig], centro_sig))
jobs2 <- con_semillas(configs[top_sigs], PARAM$semillas_etapa2, 2L)
tb_trials2 <- correr(jobs2, "trials_th04_etapa2.tsv")

agg2 <- tb_trials2[is.finite(ganancia), .(
  ganancia2 = mean(ganancia),
  sd2 = sd(ganancia),
  se2 = sd(ganancia) / sqrt(.N),
  n2 = .N,
  envios_med = as.integer(round(median(mejores_envios)))
), by = .(cfg_sig, familia, param_optim)]
agg2 <- agg1[, .(cfg_sig, ganancia1 = ganancia)][agg2, on = "cfg_sig"]
agg2[, encogimiento := ganancia2 - ganancia1]      # +++ cuanto del maximo de etapa 1 era suerte
setorder(agg2, -ganancia2)
fwrite(agg2, "configs_th04_etapa2.tsv", sep = "\t")

centro2 <- agg2[cfg_sig == centro_sig, ganancia2][1]
agg2[, delta_vs_centro := ganancia2 - centro2]
cat("=== etapa 2 (", length(PARAM$semillas_etapa2), " semillas nuevas) ===\n", sep = "")
print(agg2[, .(familia, ganancia1, ganancia2, se2, encogimiento, delta_vs_centro, envios_med)])

ganador_sig <- agg2$cfg_sig[1]
mejor <- configs[[ganador_sig]]
cat("\nGanador:", mejor$familia, "| media etapa 2:", format(agg2$ganancia2[1], big.mark = ",", scientific = FALSE),
    "| +-", format(round(agg2$se2[1]), big.mark = ","), "(1 se)\n")
if (ganador_sig == centro_sig) {
  cat("El centro sigue siendo el mejor.\n")
} else if (isTRUE(agg2$delta_vs_centro[1] < 2 * agg2$se2[1])) {
  cat("OJO: le gana al centro por menos de 2 errores estandar. Es casi un empate.\n")
}

# +++ fila de planilla: una por config de etapa 2, con sus hiperparametros (salen de configs, no de eval_job)
cfg_dt <- rbindlist(lapply(configs[agg2$cfg_sig], function(j) as.data.table(j[setdiff(names(j), "cfg_sig")])), fill = TRUE)
cfg_dt[, cfg_sig := agg2$cfg_sig]
rep_row <- agg2[cfg_dt[, !c("familia", "param_optim")], on = "cfg_sig"]
setorder(rep_row, -ganancia2)

planilla <- rep_row[, .(
  semilla_primigenia = PARAM$semilla_primigenia,
  `semilla2 (undersampling)` = paste(PARAM$semillas_etapa2 + 17L, collapse = ","),
  `Porción Undesampling` = undersampling,
  `Param a optimizar` = param_optim,
  `# semillas en BO` = n2,
  `BO iterations` = length(configs),
  num_iterations = num_iterations,
  learning_rate = learning_rate,
  feature_fraction = feature_fraction,
  min_data_in_leaf = min_data_in_leaf,
  num_leaves = num_leaves,
  max_depth = max_depth,
  lambda_l1 = lambda_l1,
  lambda_l2 = lambda_l2,
  min_gain_to_split = min_gain_to_split,
  bagging_fraction = bagging_fraction,
  min_sum_hessian_in_leaf = min_sum_hessian_in_leaf,
  bagging_freq = bagging_freq,
  scale_pos_weight = scale_pos_weight,
  boosting = boosting,
  boost_from_average = boost_from_average,
  objective = objective,
  first_metric_only = TRUE,
  drop_rate = drop_rate,
  skip_drop = skip_drop,
  force_col_wise = force_col_wise,
  is_unbalance = is_unbalance,
  max_drop = max_drop,
  max_bin = max_bin,
  n_estimators = num_iterations,
  early_stopping_rounds = early_stopping_rounds,
  min_child_weight = min_sum_hessian_in_leaf,
  feature_fraction_bynode = feature_fraction_bynode,
  bagging_freq_2 = bagging_freq,
  Prueba = ganancia2,
  `mejores envios` = envios_med,
  `Public Leaderboard` = NA_real_,
  `Promedio de las pruebas` = ganancia1,
  Estudiante = PARAM$estudiante
)]
fwrite(planilla, "planilla_TH04.tsv", sep = "\t")
cat("\nplanilla:", nrow(planilla), "filas\n")

## 8. Submit del ganador (opcional)

Entrena sobre **todo** 202107, con los mismos params que el HPO (sale de `lgb_params_de`, no se pierde nada).

`min_data_in_leaf` se escala por `1/(0.7 * undersampling)`: en el HPO el modelo veia el 70% del mes y ademas undersampleado.

Semillerio: se promedian las probabilidades de `PARAM$semillerio` semillas, asi el orden de los clientes no depende de una.

El cupo sale de la mediana de `mejores_envios` de la etapa 2, redondeada a 500. Se generan CSV de ese cupo +-1000 y se submitean 3 (cupo-500, cupo, cupo+500). No elijas el cupo por el Public: entre cupos se mueve +-10 puntos por ruido.

In [ ]:
CORRER_KAGGLE <- FALSE  # +++ pasar a TRUE cuando ya viste el ganador de la etapa 2

cupo <- as.integer(round(agg2$envios_med[1] / 500) * 500)
PARAM$submit_cortes <- cupo + c(-500L, 0L, 500L)
cat("cupo centro:", cupo, " | a submitear:", PARAM$submit_cortes, "\n")

if (CORRER_KAGGLE) {
  dfull <- dataset[foto_mes == 202107]
  dfuture <- dataset[foto_mes == 202109]
  y <- if (identical(mejor$label_mode, "baja1y2")) {
    as.integer(dfull$clase_ternaria %in% c("BAJA+1", "BAJA+2"))
  } else {
    as.integer(dfull$clase_ternaria == "BAJA+2")
  }
  drop_cols <- c("clase_ternaria", "numero_de_cliente", "foto_mes", "clase01", "azar", "training")
  campos <- setdiff(colnames(dfull), drop_cols)
  campos <- campos[vapply(dfull[, campos, with = FALSE], is.numeric, logical(1))]

  # +++ el final no se undersamplea: min_data se lleva del 70%*us al mes completo
  min_data_final <- as.integer(round(mejor$min_data_in_leaf / ((1 - PARAM$test_frac) * mejor$undersampling)))
  cat("min_data_in_leaf final:", min_data_final, "\n")

  X <- data.matrix(dfull[, campos, with = FALSE])
  Xf <- data.matrix(dfuture[, campos, with = FALSE])
  semillas_final <- c(PARAM$semillas_etapa1, PARAM$semillas_etapa2)[seq_len(PARAM$semillerio)]
  prob <- numeric(nrow(Xf))
  for (s in semillas_final) {
    ds <- lgb.Dataset(data = X, label = y, params = lgb_dataset_params_de(mejor), free_raw_data = FALSE)
    modelo <- lgb.train(
      params = lgb_params_de(mejor, num_threads = PARAM$mc_cores, min_data_in_leaf = min_data_final, seed = s),
      data = ds, nrounds = as.integer(mejor$num_iterations), verbose = -1
    )
    prob <- prob + predict(modelo, Xf) / length(semillas_final)
    cat("semilla", s, "lista\n")
    flush.console()
  }
  fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = prob), "KA497_prob.csv")
  ord <- order(prob, decreasing = TRUE)

  for (envios in seq(cupo - 1000L, cupo + 1000L, by = 500L)) {
    pred <- integer(length(prob))
    pred[ord[seq_len(envios)]] <- 1L
    archivo <- sprintf("KA497_%05d.csv", envios)
    fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, Predicted = pred), archivo)
    cat("csv", archivo, "\n")
    if (envios %in% PARAM$submit_cortes) {
      linea <- sprintf(
        "kaggle competitions submit -c labo-1-ba-inicial -f %s -m 'TH04v2 %s envios=%d leaves=%s lr=%s semillerio=%d'",
        archivo, mejor$familia, envios, mejor$num_leaves, mejor$learning_rate, length(semillas_final)
      )
      cat("  SUBMIT:", system(linea, intern = TRUE), "\n")
    }
  }
  flush.console()
}

In [ ]:
# +++ scores de Kaggle, para copiar a Public Leaderboard
if (CORRER_KAGGLE) {
  cat(system("kaggle competitions submissions -c labo-1-ba-inicial", intern = TRUE), sep = "\n")
}